# exp-017: 검증셋 500쌍 고정 + 황금 가중치 그리드 서치

- **목적:** 5관점(A/B/C/D/E)의 *임의 가중치* 표(`role 0.6 / skill 0.3 / industry 0.1` 류)가 정말 최적인지 검증. 100쌍×5관점=500쌍 검증셋에서 5차원 simplex 그리드 서치로 **황금 가중치** 산출 → 임의 vs 황금 NDCG@10 격차 정량화.
- **RQ:** RQ3 (GT 부재 환경의 평가 표준화) + 차별성 ④
- **차별성 축:** ④ GT 부재 (100쌍이 곧 고정 검증셋)
- **입력 데이터:** `raw/data/gemini_profile_outputs/benchmark_labeled_100_{A,B,C,D,E}.csv` (5관점 × 100쌍, 5 컴포넌트 score + judge_relevance 0~4)
- **출력 위치:** `raw/experiments/exp-017-validation-grid-search/`
- **관련 위키:** learnable-fusion-실험계획 §4 exp-017, exp-006-25cell-perspective-matrix
- **작성일:** 2026-05-31
- **시드:** 42
- **LLM 호출:** ❌ 없음 (자체 임베딩 + fusion 노선 정합)

## 절차

1. 검증셋 500쌍 로드, 5 컴포넌트 + judge_relevance 분포 진단
2. NDCG@10 함수 (관점별 user-mean)
3. 임의 가중치 5세트 baseline NDCG 계산
4. 5차원 simplex 그리드 생성 (해상도 0.05, 약 10k 점)
5. 그리드 서치 → 관점별 **황금 가중치** 산출
6. cross-perspective 25셀 매트릭스 (가중치 5세트 × 데이터 5관점) — exp-006 포맷
7. 결과 저장 → 다음(exp-018 학습 head)으로 인수인계

In [1]:
# 환경 / 경로 / 시드
import json, itertools, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path('.')
BENCH_DIR = ROOT / 'raw' / 'data' / 'gemini_profile_outputs'
OUT_DIR = ROOT / 'raw' / 'experiments' / 'exp-017-validation-grid-search'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PERSPECTIVES = ['A','B','C','D','E']
COMPONENT_COLS = ['role_match', 'hard_skill', 'industry_match', 'star_overlap', 'context']
COMPONENT_LABELS = ['role', 'skill', 'industry', 'star', 'context']
LABEL_COL = 'judge_relevance'

print('ROOT       :', ROOT)
print('BENCH_DIR  :', BENCH_DIR.relative_to(ROOT))
print('OUT_DIR    :', OUT_DIR.relative_to(ROOT))
print('COMPONENTS :', list(zip(COMPONENT_COLS, COMPONENT_LABELS)))


ROOT       : .
BENCH_DIR  : raw/data/gemini_profile_outputs
OUT_DIR    : raw/experiments/exp-017-validation-grid-search
COMPONENTS : [('role_match', 'role'), ('hard_skill', 'skill'), ('industry_match', 'industry'), ('star_overlap', 'star'), ('context', 'context')]


In [2]:
# === 검증셋 500쌍 로드 ===
frames = {}
for p in PERSPECTIVES:
    df = pd.read_csv(BENCH_DIR / f'benchmark_labeled_100_{p}.csv')
    df['perspective'] = p
    frames[p] = df

val_500 = pd.concat(frames.values(), ignore_index=True)
print(f'전체 검증셋: {len(val_500)} 행 (5관점 × 100쌍)')
print(f'  unique userId : {val_500["userId"].nunique()}')
print(f'  unique job_id : {val_500["job_id"].nunique()}')
print()

# 관점별 진단
diag_rows = []
for p, df in frames.items():
    rel_dist = df[LABEL_COL].value_counts().sort_index().to_dict()
    diag_rows.append({
        'perspective': p,
        'n_pairs': len(df),
        'n_users': df['userId'].nunique(),
        'n_jobs': df['job_id'].nunique(),
        'rel_0': rel_dist.get(0, 0),
        'rel_1': rel_dist.get(1, 0),
        'rel_2': rel_dist.get(2, 0),
        'rel_3': rel_dist.get(3, 0),
        'rel_4': rel_dist.get(4, 0),
        'mean_rel': df[LABEL_COL].mean(),
    })
diag = pd.DataFrame(diag_rows)
print(diag.to_string(index=False))

# 컴포넌트 score 분포
print()
print('5 컴포넌트 score 분포 (관점 무관, 전체 500쌍):')
print(val_500[COMPONENT_COLS].describe().round(3).to_string())


전체 검증셋: 500 행 (5관점 × 100쌍)
  unique userId : 49
  unique job_id : 256

perspective  n_pairs  n_users  n_jobs  rel_0  rel_1  rel_2  rel_3  rel_4  mean_rel
          A      100       10      70     18      1      4     27     50      2.90
          B      100       10      74     16     10     21     53      0      2.11
          C      100       10      72     16      7      2     20     55      2.91
          D      100       10      85     15     21      0      0     64      2.77
          E      100       10      83      8      8     26     25     33      2.67

5 컴포넌트 score 분포 (관점 무관, 전체 500쌍):
       role_match  hard_skill  industry_match  star_overlap  context
count     500.000     500.000         500.000       500.000  500.000
mean        0.740       0.548           0.614         0.148    0.614
std         0.439       0.465           0.487         0.134    0.487
min         0.000       0.000           0.000         0.000    0.000
25%         0.000       0.000           0.000      

In [3]:
# === NDCG@K 함수 (batched: 한 user의 후보들을 그리드 전체에 한 번에 적용) ===

def dcg_at_k(sorted_labels, k):
    # sorted_labels: (..., n_cand) — already sorted by predicted score desc
    k = min(k, sorted_labels.shape[-1])
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    return (sorted_labels[..., :k] * discounts).sum(axis=-1)


def ndcg_at_k_for_user(scores, labels, k=10):
    # scores: (n_cand,) or (n_grid, n_cand)
    # labels: (n_cand,)
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    if scores.ndim == 1:
        scores = scores[None, :]
        squeeze = True
    else:
        squeeze = False
    n_grid, n_cand = scores.shape
    order = np.argsort(-scores, axis=1)  # tie -> stable order
    sorted_labels = labels[order]
    dcg_vals = dcg_at_k(sorted_labels, k)
    ideal = np.sort(labels)[::-1]
    idcg = dcg_at_k(ideal[None, :], k)[0]
    out = dcg_vals / idcg if idcg > 0 else np.zeros(n_grid)
    return out[0] if squeeze else out


def perspective_ndcg(weights, df_p, k=10):
    # weights: (5,) or (n_grid, 5)
    # df_p: 한 관점 100쌍 DataFrame
    weights = np.atleast_2d(np.asarray(weights, dtype=float))
    n_grid = weights.shape[0]
    user_ndcgs = []
    for uid, sub in df_p.groupby('userId', sort=True):
        comps = sub[COMPONENT_COLS].values.astype(float)  # (n_cand, 5)
        labels = sub[LABEL_COL].values.astype(float)
        scores = weights @ comps.T  # (n_grid, n_cand)
        ndcg = ndcg_at_k_for_user(scores, labels, k=k)  # (n_grid,)
        user_ndcgs.append(ndcg)
    user_ndcgs = np.stack(user_ndcgs, axis=1)  # (n_grid, n_users)
    mean_ndcg = user_ndcgs.mean(axis=1)
    return mean_ndcg[0] if mean_ndcg.shape[0] == 1 else mean_ndcg


# === sanity check: 단일 가중치 1개로 한 관점 NDCG ===
test_w = np.array([0.6, 0.3, 0.1, 0.0, 0.0])
ndcg_A = perspective_ndcg(test_w, frames['A'], k=10)
print(f'sanity check — A 관점 / weights=[0.6,0.3,0.1,0,0] : NDCG@10 = {ndcg_A:.4f}')


sanity check — A 관점 / weights=[0.6,0.3,0.1,0,0] : NDCG@10 = 1.0000


In [4]:
# === 임의 가중치 5세트 (현행 시스템) baseline ===
# 컬럼 매핑: role / skill / industry / star / context = COMPONENT_COLS

ARBITRARY_WEIGHTS = {
    'A': {'role': 0.6, 'skill': 0.3, 'industry': 0.1, 'star': 0.0, 'context': 0.0},  # Job-Centric
    'B': {'role': 0.2, 'skill': 0.3, 'industry': 0.0, 'star': 0.5, 'context': 0.0},  # Resume-Centric
    'C': {'role': 0.2, 'skill': 0.7, 'industry': 0.1, 'star': 0.0, 'context': 0.0},  # Skill-Centric
    'D': {'role': 0.2, 'skill': 0.0, 'industry': 0.5, 'star': 0.0, 'context': 0.3},  # Context-Fit
    'E': {'role': 0.2, 'skill': 0.2, 'industry': 0.2, 'star': 0.2, 'context': 0.2},  # Mixed/Default
}

def weights_to_vec(d):
    return np.array([d['role'], d['skill'], d['industry'], d['star'], d['context']], dtype=float)

# 5 가중치 세트 × 5 관점 데이터 baseline
baseline_rows = []
for w_name, w_dict in ARBITRARY_WEIGHTS.items():
    w_vec = weights_to_vec(w_dict)
    for p in PERSPECTIVES:
        ndcg = perspective_ndcg(w_vec, frames[p], k=10)
        baseline_rows.append({
            'weight_set': w_name,
            'data_perspective': p,
            'role': w_dict['role'],
            'skill': w_dict['skill'],
            'industry': w_dict['industry'],
            'star': w_dict['star'],
            'context': w_dict['context'],
            'ndcg10': ndcg,
        })
baseline_df = pd.DataFrame(baseline_rows)
print('=== 임의 가중치 baseline ===')
print()
# 대각선 (가중치 X가 X 관점 데이터에)
diag = baseline_df[baseline_df['weight_set'] == baseline_df['data_perspective']]
print('대각선 (각 관점의 임의 가중치 적용):')
print(diag[['weight_set', 'role', 'skill', 'industry', 'star', 'context', 'ndcg10']].to_string(index=False))
print()
print(f'대각선 평균 NDCG@10 = {diag["ndcg10"].mean():.4f}')


=== 임의 가중치 baseline ===

대각선 (각 관점의 임의 가중치 적용):
weight_set  role  skill  industry  star  context   ndcg10
         A   0.6    0.3       0.1   0.0      0.0 1.000000
         B   0.2    0.3       0.0   0.5      0.0 1.000000
         C   0.2    0.7       0.1   0.0      0.0 1.000000
         D   0.2    0.0       0.5   0.0      0.3 1.000000
         E   0.2    0.2       0.2   0.2      0.2 0.999918

대각선 평균 NDCG@10 = 1.0000


In [5]:
# === 5차원 simplex 그리드 (sum=1, 해상도 0.05 → 약 10k 점) ===

def simplex_grid(n_dim, step):
    grid_size = int(round(1.0 / step))
    pts = []
    # n_dim=5
    for k1 in range(grid_size + 1):
        for k2 in range(grid_size + 1 - k1):
            for k3 in range(grid_size + 1 - k1 - k2):
                for k4 in range(grid_size + 1 - k1 - k2 - k3):
                    k5 = grid_size - k1 - k2 - k3 - k4
                    pts.append([k1, k2, k3, k4, k5])
    arr = np.asarray(pts, dtype=float) * step
    return arr  # (n_pts, n_dim)


STEP = 0.05
grid = simplex_grid(5, STEP)
print(f'simplex grid : step={STEP}, n_points={len(grid):,}')
print(f'각 점의 합 sanity: min={grid.sum(axis=1).min():.4f}, max={grid.sum(axis=1).max():.4f}')
print(f'그리드 메모리 : {grid.nbytes/1024:.1f} KB')


simplex grid : step=0.05, n_points=10,626
각 점의 합 sanity: min=1.0000, max=1.0000
그리드 메모리 : 415.1 KB


In [6]:
# === 그리드 서치: 관점별 NDCG@10 최대 가중치 = 황금 가중치 ===
import time

golden_rows = []
all_grid_ndcg = {}  # perspective -> ndarray of NDCG over grid points

for p in PERSPECTIVES:
    t0 = time.time()
    ndcg_grid = perspective_ndcg(grid, frames[p], k=10)  # (n_grid,)
    all_grid_ndcg[p] = ndcg_grid
    best_idx = int(np.argmax(ndcg_grid))
    best_w = grid[best_idx]
    best_ndcg = float(ndcg_grid[best_idx])
    golden_rows.append({
        'perspective': p,
        'role': best_w[0], 'skill': best_w[1], 'industry': best_w[2],
        'star': best_w[3], 'context': best_w[4],
        'golden_ndcg10': best_ndcg,
        'n_grid': len(grid),
        'search_sec': round(time.time() - t0, 2),
    })
    print(f'  {p}: golden NDCG@10={best_ndcg:.4f} | weights={best_w.round(3).tolist()} | {time.time()-t0:.1f}s')

golden_df = pd.DataFrame(golden_rows)
print()
print('=== 황금 가중치 (관점별 최대 NDCG@10 weights) ===')
print(golden_df.to_string(index=False))


  A: golden NDCG@10=1.0000 | weights=[0.4, 0.1, 0.0, 0.45, 0.05] | 0.0s
  B: golden NDCG@10=1.0000 | weights=[0.2, 0.3, 0.0, 0.5, 0.0] | 0.0s
  C: golden NDCG@10=1.0000 | weights=[0.0, 0.9, 0.0, 0.0, 0.1] | 0.0s
  D: golden NDCG@10=1.0000 | weights=[0.0, 0.0, 0.0, 0.0, 1.0] | 0.0s
  E: golden NDCG@10=1.0000 | weights=[0.05, 0.25, 0.0, 0.45, 0.25] | 0.0s

=== 황금 가중치 (관점별 최대 NDCG@10 weights) ===
perspective  role  skill  industry  star  context  golden_ndcg10  n_grid  search_sec
          A  0.40   0.10       0.0  0.45     0.05            1.0   10626        0.01
          B  0.20   0.30       0.0  0.50     0.00            1.0   10626        0.01
          C  0.00   0.90       0.0  0.00     0.10            1.0   10626        0.01
          D  0.00   0.00       0.0  0.00     1.00            1.0   10626        0.01
          E  0.05   0.25       0.0  0.45     0.25            1.0   10626        0.01


In [7]:
# === 임의 vs 황금 비교 (대각선) ===
arb_diag = baseline_df[baseline_df['weight_set'] == baseline_df['data_perspective']].copy()
arb_diag = arb_diag.set_index('weight_set')[['ndcg10']].rename(columns={'ndcg10': 'arbitrary_ndcg10'})

cmp = golden_df.set_index('perspective').join(arb_diag)
cmp['delta'] = cmp['golden_ndcg10'] - cmp['arbitrary_ndcg10']
cmp = cmp[['arbitrary_ndcg10', 'golden_ndcg10', 'delta', 'role', 'skill', 'industry', 'star', 'context']]
print('=== 임의 가중치 vs 황금 가중치 ===')
print(cmp.round(4).to_string())
print()
print(f'평균 NDCG@10 격차 (golden − arbitrary) = {cmp["delta"].mean():+.4f}p')
print(f'성공 기준 (평균 격차 ≥ +0.02p) : {"✅ 통과" if cmp["delta"].mean() >= 0.02 else "❌ 미달 (임의 가중치가 이미 충분)"}')
print(f'관점 중 황금 ≠ 임의(격차>0.001) : {(cmp["delta"] > 0.001).sum()} / 5')


=== 임의 가중치 vs 황금 가중치 ===
             arbitrary_ndcg10  golden_ndcg10   delta  role  skill  industry  star  context
perspective                                                                               
A                      1.0000            1.0  0.0000  0.40   0.10       0.0  0.45     0.05
B                      1.0000            1.0  0.0000  0.20   0.30       0.0  0.50     0.00
C                      1.0000            1.0  0.0000  0.00   0.90       0.0  0.00     0.10
D                      1.0000            1.0  0.0000  0.00   0.00       0.0  0.00     1.00
E                      0.9999            1.0  0.0001  0.05   0.25       0.0  0.45     0.25

평균 NDCG@10 격차 (golden − arbitrary) = +0.0000p
성공 기준 (평균 격차 ≥ +0.02p) : ❌ 미달 (임의 가중치가 이미 충분)
관점 중 황금 ≠ 임의(격차>0.001) : 0 / 5


In [8]:
# === Cross-perspective 25셀 매트릭스 (exp-006 포맷) ===
# 행 = 가중치 세트(임의 5 + 황금 5), 열 = 데이터 관점

# 가중치 세트 정의
weight_sets = {}
for k, v in ARBITRARY_WEIGHTS.items():
    weight_sets[f'arb_{k}'] = weights_to_vec(v)
for _, row in golden_df.iterrows():
    weight_sets[f'gold_{row["perspective"]}'] = np.array(
        [row['role'], row['skill'], row['industry'], row['star'], row['context']]
    )

matrix_rows = []
for w_name, w_vec in weight_sets.items():
    row = {'weight_set': w_name}
    for p in PERSPECTIVES:
        row[p] = perspective_ndcg(w_vec, frames[p], k=10)
    row['mean'] = np.mean([row[p] for p in PERSPECTIVES])
    matrix_rows.append(row)

matrix_df = pd.DataFrame(matrix_rows).set_index('weight_set')
print('=== Cross-perspective 25셀 (행=가중치, 열=데이터 관점) ===')
print(matrix_df.round(4).to_string())
print()

# 임의 vs 황금 평균 비교
arb_mean = matrix_df.loc[[f'arb_{p}' for p in PERSPECTIVES], 'mean'].mean()
gold_mean = matrix_df.loc[[f'gold_{p}' for p in PERSPECTIVES], 'mean'].mean()
print(f'임의 5세트 평균(평균행)  : {arb_mean:.4f}')
print(f'황금 5세트 평균(평균행)  : {gold_mean:.4f}')
print(f'전체 격차                 : {gold_mean - arb_mean:+.4f}p')


=== Cross-perspective 25셀 (행=가중치, 열=데이터 관점) ===
                 A       B       C       D       E    mean
weight_set                                                
arb_A       1.0000  0.9712  0.9648  0.9415  0.9992  0.9753
arb_B       0.9929  1.0000  0.9944  0.9047  0.9890  0.9762
arb_C       0.9845  0.9914  1.0000  0.9395  0.9992  0.9829
arb_D       0.9836  0.9681  0.9519  1.0000  0.9997  0.9807
arb_E       0.9845  0.9692  0.9670  0.9956  0.9999  0.9832
gold_A      1.0000  0.9801  0.9499  0.9293  0.9879  0.9694
gold_B      0.9929  1.0000  0.9944  0.9047  0.9890  0.9762
gold_C      0.9835  0.9911  1.0000  0.9409  0.9991  0.9829
gold_D      0.9836  0.9732  0.9907  1.0000  0.9994  0.9894
gold_E      0.9784  0.9854  0.9913  0.9710  1.0000  0.9852

임의 5세트 평균(평균행)  : 0.9797
황금 5세트 평균(평균행)  : 0.9806
전체 격차                 : +0.0010p


In [9]:
# === 추가 메트릭: NDCG@5 / Recall@5 / MRR@10 ===
def metric_at_k_for_user(scores, labels, k, kind):
    labels = np.asarray(labels, dtype=float)
    order = np.argsort(-scores)
    sl = labels[order]
    if kind == 'ndcg':
        return float(dcg_at_k(sl[None, :], k)[0] / max(dcg_at_k(np.sort(labels)[::-1][None, :], k)[0], 1e-9))
    if kind == 'recall':
        rel_mask = labels >= 3  # graded≥3 = relevant
        if rel_mask.sum() == 0:
            return 0.0
        hit = (sl[:k] >= 3).sum()
        return float(hit / rel_mask.sum())
    if kind == 'mrr':
        # graded≥3 첫 등장 위치
        ranks = np.where(sl[:k] >= 3)[0]
        return float(1.0 / (ranks[0] + 1)) if len(ranks) else 0.0
    raise ValueError(kind)


def perspective_metrics(weights, df_p, k=10):
    w = np.atleast_1d(weights).astype(float)
    out = {'ndcg10': [], 'ndcg5': [], 'recall5': [], 'mrr10': []}
    for uid, sub in df_p.groupby('userId', sort=True):
        comps = sub[COMPONENT_COLS].values.astype(float)
        labels = sub[LABEL_COL].values.astype(float)
        scores = comps @ w
        out['ndcg10'].append(metric_at_k_for_user(scores, labels, 10, 'ndcg'))
        out['ndcg5'].append(metric_at_k_for_user(scores, labels, 5, 'ndcg'))
        out['recall5'].append(metric_at_k_for_user(scores, labels, 5, 'recall'))
        out['mrr10'].append(metric_at_k_for_user(scores, labels, 10, 'mrr'))
    return {k_: float(np.mean(v_)) for k_, v_ in out.items()}


multi_rows = []
for w_name, w_vec in weight_sets.items():
    for p in PERSPECTIVES:
        m = perspective_metrics(w_vec, frames[p], k=10)
        multi_rows.append({'weight_set': w_name, 'perspective': p, **m})

multi_df = pd.DataFrame(multi_rows)
print('=== 보조 메트릭 (대각선만) ===')
diag_multi = multi_df[multi_df.apply(
    lambda r: r['weight_set'].split('_')[-1] == r['perspective'], axis=1
)].copy()
diag_multi['type'] = diag_multi['weight_set'].str.split('_').str[0]
print(diag_multi.pivot_table(
    index='perspective', columns='type', values=['ndcg10', 'ndcg5', 'recall5', 'mrr10']
).round(4).to_string())


=== 보조 메트릭 (대각선만) ===


            mrr10       ndcg10      ndcg5      recall5        
type          arb gold     arb gold   arb gold     arb    gold
perspective                                                   
A             1.0  1.0  1.0000  1.0   1.0  1.0  0.6617  0.6617
B             0.9  0.9  1.0000  1.0   1.0  1.0  0.6889  0.6889
C             1.0  1.0  1.0000  1.0   1.0  1.0  0.6776  0.6776
D             0.9  0.9  1.0000  1.0   1.0  1.0  0.6478  0.6478
E             1.0  1.0  0.9999  1.0   1.0  1.0  0.7782  0.7782


In [10]:
# === 결과 저장 ===
import json as _json

# 1. baseline (임의 가중치 25셀)
baseline_df.to_csv(OUT_DIR / 'baseline_arbitrary_25cell.csv', index=False)

# 2. golden weights
golden_df.to_csv(OUT_DIR / 'golden_weights.csv', index=False)

# 3. 임의 vs 황금 비교
cmp.to_csv(OUT_DIR / 'arbitrary_vs_golden.csv')

# 4. cross-perspective 25셀 매트릭스 (10 세트 × 5 관점)
matrix_df.to_csv(OUT_DIR / 'cross_perspective_matrix.csv')

# 5. 보조 메트릭 (NDCG@5 / Recall@5 / MRR@10)
multi_df.to_csv(OUT_DIR / 'multi_metrics.csv', index=False)

# 6. 전체 그리드 NDCG (관점별, exp-018 학습 head의 pseudo-supervision 재사용)
grid_dump = pd.DataFrame(grid, columns=COMPONENT_LABELS)
for p in PERSPECTIVES:
    grid_dump[f'ndcg10_{p}'] = all_grid_ndcg[p]
grid_dump.to_csv(OUT_DIR / 'grid_full.csv.gz', index=False, compression='gzip')

# 7. 메타 / sanity
meta = {
    'experiment': 'exp-017-validation-grid-search',
    'date': '2026-05-31',
    'seed': RANDOM_SEED,
    'n_validation_pairs': int(len(val_500)),
    'perspectives': PERSPECTIVES,
    'component_cols': COMPONENT_COLS,
    'component_labels': COMPONENT_LABELS,
    'simplex_step': STEP,
    'n_grid_points': int(len(grid)),
    'arbitrary_diag_mean_ndcg10': float(arb_diag['arbitrary_ndcg10'].mean()),
    'golden_diag_mean_ndcg10': float(golden_df['golden_ndcg10'].mean()),
    'mean_delta': float(cmp['delta'].mean()),
    'arbitrary_weights': ARBITRARY_WEIGHTS,
    'golden_weights': {
        r['perspective']: {
            'role': r['role'], 'skill': r['skill'], 'industry': r['industry'],
            'star': r['star'], 'context': r['context'],
            'ndcg10': r['golden_ndcg10'],
        }
        for _, r in golden_df.iterrows()
    },
}
(OUT_DIR / 'meta.json').write_text(_json.dumps(meta, ensure_ascii=False, indent=2))

print('=== 저장 완료 ===')
for f in sorted(OUT_DIR.iterdir()):
    sz = f.stat().st_size
    print(f'  {f.name:<35s} {sz/1024:>8.1f} KB')


=== 저장 완료 ===
  arbitrary_vs_golden.csv                  0.3 KB
  baseline_arbitrary_25cell.csv            1.1 KB
  cross_perspective_matrix.csv             1.1 KB
  golden_weights.csv                       0.3 KB
  grid_full.csv.gz                        95.2 KB
  meta.json                                1.9 KB
  multi_metrics.csv                        3.3 KB


## 결과 해석 가이드 (수동 단계)

이 노트북을 끝내고 나면:

1. **`golden_weights.csv`** — 5관점별 6차원 황금 가중치. 임의 가중치 표(`role 0.6 / skill 0.3 / industry 0.1` 류)와 직접 비교 가능
2. **`arbitrary_vs_golden.csv`** — 관점별 NDCG@10 격차. **평균 격차 ≥ +0.02p** 이면 learnable-fusion-실험계획 exp-017 성공 기준 통과
3. **`cross_perspective_matrix.csv`** — exp-006 25셀 패턴 확장. 황금 가중치도 *자기 관점에서만* 1위인지(분기 효과 검증), 다른 관점에서도 강건한지 확인
4. **`grid_full.parquet`** — 전체 그리드 결과. exp-018 학습 head의 *pseudo-supervision* 신호 원천으로 재사용

## 다음 단계 (exp-018)

- 학습 데이터 라벨 방식 결정 — (a) `interestedJobs ↔ jd_job_role` 자동 매칭 (b) Model C 6 컴포넌트 pseudo-relevance
- V1 (Logistic Regression) 베이스라인 → V2 (BERT+Linear head) → V3 (5관점 분기 head)
- 학습 head NDCG@10 ≥ 본 노트북 황금 가중치 NDCG@10 인지 검증

## 출력 위치

- `raw/experiments/exp-017-validation-grid-search/` 전체

## 관련

- learnable-fusion-실험계획 §4 exp-017
- exp-006-25cell-perspective-matrix — cross-perspective 25셀 원형
- 관점별-fusion-가중치-설계 — 임의 가중치 5세트 정의